# 📄 NEUMOALERT — Notebook 2: NLP Clínico en 4 Categorías (Sección E) + Fusión Multimodal

**Proyecto Capstone · SIC 2026 IA**  
Clasifica informes radiológicos PDF en las **4 categorías exactas de la Sección E del estudio del médico**:
1. `Sin neumonía`
2. `Posible`
3. `Probable`
4. `Definitiva`

Y luego muestra cómo se une con la **Red Neuronal (CNN)** y el **Random Forest (ficha Heckerling)**.


In [ ]:
!pip install pymupdf scikit-learn pandas matplotlib joblib -q
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

CSV_CANDIDATES = ["radiology_reports_dataset.csv", "../data/radiology_reports_dataset.csv"]
csv_found = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)
assert csv_found is not None, "Sube el archivo radiology_reports_dataset.csv de la carpeta data/ a Colab."

df_rx = pd.read_csv(csv_found)
print("Informes:", len(df_rx))
print(df_rx["categoria_rx"].value_counts())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_rx["texto_informe"], df_rx["categoria_rx"], test_size=0.20, random_state=42, stratify=df_rx["categoria_rx"]
)

nlp_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 3), min_df=2, max_features=3500, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, C=2.2, class_weight="balanced", random_state=42)),
])
nlp_pipe.fit(X_train, y_train)
y_pred = nlp_pipe.predict(X_test)

orden = ["Sin neumonía", "Posible", "Probable", "Definitiva"]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%} | F1 Macro: {f1_score(y_test, y_pred, average='macro'):.1%}\n")
print(classification_report(y_test, y_pred, labels=orden))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=orden, cmap="Greens")
plt.title("Matriz de Confusión 4×4 — Categorías del Radiólogo (Sección E)")
plt.tight_layout()
plt.show()


In [ ]:
# Prueba rápida con ejemplos reales (incluyendo negación trampa)
ejemplos = [
    "Motivo: descartar neumonía. Hallazgos: Consolidación homogénea con broncograma aéreo en base derecha. Conclusión: Infiltrado neumónico inequívoco.",
    "Hallazgos: Opacidad alveolar parcheada basal izquierda. Conclusión: Lo más probable es neumonía adquirida en la comunidad.",
    "Hallazgos: Tenue infiltrado intersticial dudoso. Conclusión: Posible infiltrado inicial, no se puede descartar neumonía.",
    "Motivo: sospecha de neumonía, descartar infiltrados. Hallazgos: Campos pulmonares limpios, sin consolidaciones. Conclusión: Estudio normal, sin neumonía.",
]
for txt in ejemplos:
    pred = nlp_pipe.predict([txt])[0]
    print(f"[{pred:12s}] -> {txt[:100]}...")
